In [2]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [3]:
fpath = 'C:\\Users\\dc00955\\OneDrive - University of Surrey\\Desktop\\Sara_case_report\\'

In [ ]:
raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_210922_260324.mtn')

In [ ]:
raw.name

In [ ]:
raw.start_time

In [ ]:
raw.duration()

In [ ]:
raw.frequency

VISUALISING

In [ ]:
layout = go.Layout(
    title="Actigraphy data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Counts/period"),
    showlegend=False
)
go.Figure(data=[go.Scatter(x=raw.data.index.astype(str), y=raw.data)], layout=layout)

MASKING INACTIVE DATA

In [8]:
raw.light.create_light_mask()

In [9]:
# manually adding mask to more than one period
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'}]
for period in periods:
    raw.add_mask_period(start=period['start'], stop=period['stop'])

In [ ]:
# visualising mask:
go.Figure(data=go.Scatter(x=raw.mask.index.astype(str),y=raw.mask),layout=layout)

In [10]:
# applying the mask:
raw.light.apply_mask = True

DAILY ACTIVITY PROFILE (create one for UK and one for Italy?)

In [31]:
layout.update(title="Daily activity profile",xaxis=dict(title="Date time"), showlegend=False);

In [32]:
daily_profile = raw.average_daily_activity(freq='15min', cyclic=False, binarize=False)

In [ ]:
go.Figure(data=[
    go.Scatter(x=daily_profile.index.astype(str), y=daily_profile)
], layout=layout)

In [ ]:
# Activity onset:
raw.AonT(freq='15min', binarize=True)

In [ ]:
# Activity offset:
raw.AoffT(freq='15min', binarize=True)

DETECTING SLEEP PERIODS

In [ ]:
# all algorithms return a binary time serie

In [11]:
layout = go.Layout(title="Rest/Activity detection",xaxis=dict(title="Date time"), yaxis=dict(title="Counts/period"), showlegend=False)

In [16]:
roenneberg = raw.Roenneberg()
roenneberg_thr = raw.Roenneberg(threshold=0.25, min_seed_period='15min') # TRESHOLD = fraction of the trend to use as a treshold for sleep/wake categorisation. min_seed_period = minimum time period required to identify a potential sleep onset

Roenneberg algorithm = detects consolidated periods of similar activity patterns. It's treshold-based but uses correlations with test series of various lengths to find the consolidated period that best matches the data

In [ ]:
go.Figure(data=[
    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
    go.Scatter(x=roenneberg.index.astype(str),y=roenneberg, yaxis='y2', name='Roenneberg'),
    go.Scatter(x=roenneberg_thr.index.astype(str),y=roenneberg_thr, yaxis='y2', name='Roenneberg (Thr:0.25)')
], layout=layout)

COLE-KRIPKE ALGORITHM = epoch-by-epoch rest-activity scoring. Uses a rolling window over the data and convolute them with a "gaussian"-like kernel. If the resulting data is above a predefined treshold, classify as activity

In [18]:
CK = raw.CK()

In [ ]:
layout.update(yaxis2=dict(title='Classification',overlaying='y',side='right'), showlegend=True);
go.Figure(data=[
    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
    go.Scatter(x=CK.index.astype(str),y=CK, yaxis='y2', name='CK')
], layout=layout)

VISUALISING LIGHT DATA

In [ ]:
raw.light

In [ ]:
raw.light.get_channel_list()

In [ ]:
# LIGHT + ACTIVITY
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Light intensity',overlaying='y',side='right'),
    showlegend=True
)

fig1 = go.Figure([
    go.Scatter(
        x=raw.data.index.astype(str),
        y=raw.data,
        name='Activity'),
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        yaxis='y2', opacity=0.5,
        name='Light')
], layout=layout)

fig1.show()